# Agentic AI with OpenAI API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/oleksa-kosovan/agent_practice/blob/main/01-Function_Calling/01-Agentic_AI_OpenAI_Eng.ipynb
)

[![Open in GitHub](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](
https://github.com/oleksa-kosovan/agent_practice/blob/main/01-Function_Calling/01-Agentic_AI_OpenAI_Eng.ipynb
)

This notebook is a hands-on guide to building **agentic AI systems** with the OpenAI API. We cover:

1. **Function Calling** — let the model invoke your Python functions
2. **Parallel Tool Calls** — handle multiple tool invocations in one turn
3. **Agentic Loop** — multi-step reasoning with iterative tool use
4. **Structured Outputs** — get typed, validated responses via Pydantic
5. **OpenAI Agents SDK** — build agents with the official SDK
6. **Multi-Agent Systems** — handoffs between specialized agents
7. **Guardrails** — input/output safety checks
8. **Agent Context & State** — share state across tools
9. **Tracing & Observability** — monitor agent execution

> **Prerequisites:** An OpenAI API key. All examples use `gpt-4o-mini` to keep costs low.

## Setup

In [1]:
!pip install -q openai openai-agents pydantic


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
from getpass import getpass

# Works in Colab, Jupyter, or terminal
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

print("✓ OpenAI client ready")

✓ OpenAI client ready


---
## 1. Function Calling Fundamentals

Function Calling lets the model decide **when** to invoke external tools and **what arguments** to pass. The model never executes code — it outputs a structured request that *your code* fulfills.

**Flow:**
1. You send a user message + tool definitions to the API
2. The model returns a `tool_calls` array (or a plain text answer)
3. Your code executes each tool call and sends results back
4. The model generates the final answer incorporating tool outputs

### 1.1 Define Python Functions

In [3]:
def get_weather(city: str, unit: str = "celsius") -> dict:
    """Simulate a weather API lookup."""
    weather_data = {
        "paris": {"temp": 18, "condition": "partly cloudy"},
        "tokyo": {"temp": 26, "condition": "sunny"},
        "new york": {"temp": 22, "condition": "rainy"},
        "london": {"temp": 14, "condition": "foggy"},
    }
    data = weather_data.get(city.lower(), {"temp": 20, "condition": "unknown"})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9 / 5 + 32)
    return {"city": city, "unit": unit, **data}


def calculate(expression: str) -> dict:
    """Safely evaluate a math expression."""
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return {"error": "Invalid characters in expression"}
    try:
        result = eval(expression)  # safe: only digits and operators
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"error": str(e)}


def search_products(query: str, max_results: int = 3) -> dict:
    """Simulate a product database search."""
    catalog = [
        {"name": "Wireless Mouse", "price": 29.99, "category": "electronics"},
        {"name": "Mechanical Keyboard", "price": 89.99, "category": "electronics"},
        {"name": "USB-C Hub", "price": 45.00, "category": "electronics"},
        {"name": "Standing Desk", "price": 399.00, "category": "furniture"},
        {"name": "Monitor Arm", "price": 79.99, "category": "furniture"},
        {"name": "Noise-Cancelling Headphones", "price": 199.99, "category": "electronics"},
    ]
    matches = [p for p in catalog if query.lower() in p["name"].lower() or query.lower() in p["category"]]
    return {"query": query, "results": matches[:max_results]}


TOOL_FUNCTIONS = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_products": search_products,
}

print("✓ Tool functions defined")

✓ Tool functions defined


### 1.2 Tool Schemas for the API

Each tool is described with a JSON Schema so the model knows **what the function does** and **what arguments it accepts**.

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city. Returns temperature and condition.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. 'Paris' or 'New York'.",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit. Defaults to celsius.",
                    },
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression. Supports +, -, *, /, parentheses.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression to evaluate, e.g. '(2 + 3) * 4'.",
                    },
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search a product catalog by keyword. Returns matching products with prices.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search keyword, e.g. 'keyboard' or 'electronics'.",
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results to return. Defaults to 3.",
                    },
                },
                "required": ["query"],
            },
        },
    },
]

print(f"✓ {len(tools)} tool schemas defined")

✓ 3 tool schemas defined


### 1.3 Single Tool Call

In [5]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
    {"role": "user", "content": "What's the weather like in Tokyo?"},
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

assistant_msg = response.choices[0].message
print("Model response:")
print(f"  Content: {assistant_msg.content}")
print(f"  Tool calls: {assistant_msg.tool_calls}")

Model response:
  Content: None
  Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_pNu4B18M3cWN0Zq8K0P0Rh4i', function=Function(arguments='{"city":"Tokyo"}', name='get_weather'), type='function')]


The model chose to call `get_weather` instead of answering directly. Now we execute the tool and send the result back:

In [6]:
# Execute the tool call
tool_call = assistant_msg.tool_calls[0]
func_name = tool_call.function.name
func_args = json.loads(tool_call.function.arguments)

print(f"Calling: {func_name}({func_args})")
result = TOOL_FUNCTIONS[func_name](**func_args)
print(f"Result: {result}")

# Send tool result back to the model
messages.append(assistant_msg)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(result),
})

final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
)

print(f"\nFinal answer: {final_response.choices[0].message.content}")

Calling: get_weather({'city': 'Tokyo'})
Result: {'city': 'Tokyo', 'unit': 'celsius', 'temp': 26, 'condition': 'sunny'}

Final answer: The weather in Tokyo is currently sunny with a temperature of 26°C.


### 1.4 Parallel Tool Calls

The model can request **multiple tools at once** when independent pieces of information are needed. This is more efficient than sequential calls.

In [7]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
    {"role": "user", "content": "Compare the weather in Paris and London, and also calculate 15% tip on a $85 bill."},
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

assistant_msg = response.choices[0].message
print(f"Model requested {len(assistant_msg.tool_calls)} tool calls:\n")

# Execute ALL tool calls
messages.append(assistant_msg)

for tc in assistant_msg.tool_calls:
    func_name = tc.function.name
    func_args = json.loads(tc.function.arguments)
    result = TOOL_FUNCTIONS[func_name](**func_args)
    print(f"  {func_name}({func_args}) → {result}")
    messages.append({
        "role": "tool",
        "tool_call_id": tc.id,
        "content": json.dumps(result),
    })

# Get the final answer
final_response = client.chat.completions.create(model=MODEL, messages=messages)
print(f"\nFinal answer:\n{final_response.choices[0].message.content}")

Model requested 3 tool calls:

  get_weather({'city': 'Paris'}) → {'city': 'Paris', 'unit': 'celsius', 'temp': 18, 'condition': 'partly cloudy'}
  get_weather({'city': 'London'}) → {'city': 'London', 'unit': 'celsius', 'temp': 14, 'condition': 'foggy'}
  calculate({'expression': '85 * 0.15'}) → {'expression': '85 * 0.15', 'result': 12.75}

Final answer:
The current weather in Paris is partly cloudy with a temperature of 18°C. In contrast, London is experiencing foggy conditions and has a temperature of 14°C. 

Regarding your bill, a 15% tip on an $85 bill would amount to $12.75.


### 1.5 Agentic Loop (Multi-Step Reasoning)

For complex tasks, the model may need to call tools **iteratively** — using the output of one tool to decide the next action. This creates an **agentic loop**.

In [8]:
def run_agent_loop(user_query: str, max_iterations: int = 5) -> str:
    """Run an agentic loop: model ↔ tools until the model produces a final answer."""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant with access to tools. "
            "Break complex problems into steps, using tools as needed. "
            "When you have enough information, give the final answer."
        )},
        {"role": "user", "content": user_query},
    ]

    for i in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        assistant_msg = response.choices[0].message

        if not assistant_msg.tool_calls:
            print(f"  [Iteration {i+1}] No tool calls → final answer")
            return assistant_msg.content

        messages.append(assistant_msg)
        for tc in assistant_msg.tool_calls:
            func_name = tc.function.name
            func_args = json.loads(tc.function.arguments)
            result = TOOL_FUNCTIONS[func_name](**func_args)
            print(f"  [Iteration {i+1}] {func_name}({func_args}) → {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })

    return "Max iterations reached."


print("Multi-step query:\n")
answer = run_agent_loop(
    "I want to buy electronics under $50. Also, what's the total if I buy "
    "a Wireless Mouse and a USB-C Hub? And is it warmer in Paris or Tokyo right now?"
)
print(f"\nAnswer:\n{answer}")

Multi-step query:

  [Iteration 1] search_products({'query': 'electronics', 'max_results': 5}) → {'query': 'electronics', 'results': [{'name': 'Wireless Mouse', 'price': 29.99, 'category': 'electronics'}, {'name': 'Mechanical Keyboard', 'price': 89.99, 'category': 'electronics'}, {'name': 'USB-C Hub', 'price': 45.0, 'category': 'electronics'}, {'name': 'Noise-Cancelling Headphones', 'price': 199.99, 'category': 'electronics'}]}
  [Iteration 1] calculate({'expression': 'price_of_wireless_mouse + price_of_usb_c_hub'}) → {'error': 'Invalid characters in expression'}
  [Iteration 1] get_weather({'city': 'Paris'}) → {'city': 'Paris', 'unit': 'celsius', 'temp': 18, 'condition': 'partly cloudy'}
  [Iteration 1] get_weather({'city': 'Tokyo'}) → {'city': 'Tokyo', 'unit': 'celsius', 'temp': 26, 'condition': 'sunny'}
  [Iteration 2] calculate({'expression': '29.99 + 45.0'}) → {'expression': '29.99 + 45.0', 'result': 74.99}
  [Iteration 3] No tool calls → final answer

Answer:
Here are the results

---
## 2. Structured Outputs

Instead of free-form text, you can ask the model to return responses that match a **Pydantic model**. This guarantees valid, typed JSON output.

In [9]:
from pydantic import BaseModel
from typing import Optional


class WeatherReport(BaseModel):
    city: str
    temperature: float
    unit: str
    condition: str
    recommendation: str


class ProductComparison(BaseModel):
    product_a: str
    product_b: str
    price_difference: float
    winner: str
    reason: str

In [10]:
response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a weather reporter. Always respond in the given format."},
        {"role": "user", "content": "Report on the weather in Paris: 18°C, partly cloudy."},
    ],
    response_format=WeatherReport,
)

report = response.choices[0].message.parsed
print(f"City: {report.city}")
print(f"Temperature: {report.temperature}°{report.unit}")
print(f"Condition: {report.condition}")
print(f"Recommendation: {report.recommendation}")
print(f"\nFull object: {report}")

City: Paris
Temperature: 18.0°°C
Condition: Partly Cloudy
Recommendation: A light jacket is recommended for the evening.

Full object: city='Paris' temperature=18.0 unit='°C' condition='Partly Cloudy' recommendation='A light jacket is recommended for the evening.'


You can also combine structured outputs with function calling — the model calls tools, then formats the final response as a Pydantic object. This is useful for building reliable data pipelines.

---
## 3. OpenAI Agents SDK

The [Agents SDK](https://github.com/openai/openai-agents-python) (`openai-agents`) provides a higher-level abstraction for building agentic systems:

- **`Agent`** — an LLM with instructions, tools, and optional output schema
- **`Runner`** — orchestrates the agent loop (tool calls, handoffs, guardrails)
- **`@function_tool`** — decorator to turn any Python function into a tool
- **Handoffs** — transfer control between specialized agents
- **Guardrails** — validate inputs and outputs

It manages the agentic loop, message history, and tool execution for you.

### 3.1 A Simple Agent

In [11]:
from agents import Agent, Runner, function_tool


@function_tool
def get_weather_tool(city: str, unit: str = "celsius") -> str:
    """Get current weather for a city. Returns temperature and condition."""
    result = get_weather(city, unit)
    return json.dumps(result)


@function_tool
def calculate_tool(expression: str) -> str:
    """Evaluate a mathematical expression. Supports +, -, *, /, parentheses."""
    result = calculate(expression)
    return json.dumps(result)


@function_tool
def search_products_tool(query: str, max_results: int = 3) -> str:
    """Search a product catalog by keyword."""
    result = search_products(query, max_results)
    return json.dumps(result)


assistant = Agent(
    name="Assistant",
    instructions=(
        "You are a helpful assistant with access to weather, calculator, "
        "and product search tools. Use them when relevant."
    ),
    tools=[get_weather_tool, calculate_tool, search_products_tool],
    model=MODEL,
)

print(f"✓ Agent '{assistant.name}' created with {len(assistant.tools)} tools")

✓ Agent 'Assistant' created with 3 tools


### 3.2 Running the Agent

In [12]:
result = await Runner.run(assistant, "What's the weather in New York?")
print(f"Final output:\n{result.final_output}")

Final output:
The weather in New York is currently 22°C and rainy.


In [13]:
result = await Runner.run(
    assistant,
    "Find me electronics under $100 and calculate the total for the cheapest two items."
)
print(f"Final output:\n{result.final_output}")

Final output:
Here are the electronics under $100:

1. **Wireless Mouse** - $29.99
2. **Mechanical Keyboard** - $89.99
3. **USB-C Hub** - $45.00

The total for the cheapest two items (Wireless Mouse and USB-C Hub) is **$74.99**.


### 3.3 Agent with Structured Output

You can set `output_type` to a Pydantic model — the agent will always return structured data:

In [14]:
class TravelBrief(BaseModel):
    city: str
    weather_summary: str
    packing_tip: str
    fun_fact: str


travel_agent = Agent(
    name="Travel Advisor",
    instructions="You help travelers prepare for trips. Use the weather tool, then give advice.",
    tools=[get_weather_tool],
    output_type=TravelBrief,
    model=MODEL,
)

result = await Runner.run(travel_agent, "I'm visiting Tokyo next week.")
brief = result.final_output_as(TravelBrief)
print(f"City: {brief.city}")
print(f"Weather: {brief.weather_summary}")
print(f"Packing tip: {brief.packing_tip}")
print(f"Fun fact: {brief.fun_fact}")

City: Tokyo
Weather: The weather in Tokyo next week is expected to be sunny with a temperature around 26°C.
Packing tip: Pack lightweight clothing and sunglasses, but also consider bringing a light jacket for cooler evenings.
Fun fact: Did you know? Tokyo is home to the world's busiest pedestrian crossing, Shibuya Crossing, where thousands of people cross the street at the same time!


---
## 4. Multi-Agent Systems with Handoffs

Real-world tasks often require **specialized knowledge**. Instead of one mega-agent, you can create multiple focused agents that **hand off** to each other.

**Pattern: Triage → Specialists**

```
User → Triage Agent → ┬→ Math Agent
                       ├→ Shopping Agent
                       └→ Weather Agent
```

Each specialist has its own instructions and tools. The triage agent decides which specialist should handle the request.

### 4.1 Define Specialized Agents

In [15]:
math_agent = Agent(
    name="Math Expert",
    instructions=(
        "You are a math expert. Help users with calculations and math problems. "
        "Use the calculator tool for arithmetic. Explain your reasoning step by step."
    ),
    tools=[calculate_tool],
    model=MODEL,
)

shopping_agent = Agent(
    name="Shopping Assistant",
    instructions=(
        "You are a shopping assistant. Help users find products, compare prices, "
        "and make purchase decisions. Use the product search tool."
    ),
    tools=[search_products_tool, calculate_tool],
    model=MODEL,
)

weather_agent = Agent(
    name="Weather Expert",
    instructions=(
        "You are a weather expert. Provide weather information, forecasts, "
        "and clothing recommendations. Use the weather tool."
    ),
    tools=[get_weather_tool],
    model=MODEL,
)

print("✓ Specialist agents created: Math, Shopping, Weather")

✓ Specialist agents created: Math, Shopping, Weather


### 4.2 Triage Agent with Handoffs

In [16]:
triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are a triage agent that routes user requests to the right specialist:\n"
        "- Math or calculation questions → hand off to Math Expert\n"
        "- Shopping, product, or price questions → hand off to Shopping Assistant\n"
        "- Weather questions → hand off to Weather Expert\n"
        "\n"
        "If the query spans multiple topics, pick the most relevant specialist. "
        "Always hand off — do not answer directly."
    ),
    handoffs=[math_agent, shopping_agent, weather_agent],
    model=MODEL,
)

print(f"✓ Triage agent created with {len(triage_agent.handoffs)} handoffs")

✓ Triage agent created with 3 handoffs


### 4.3 Run the Multi-Agent System

In [17]:
queries = [
    "What is 17 * 23 + 45?",
    "I need a good keyboard under $100",
    "Should I bring an umbrella to London today?",
]

for query in queries:
    print(f"User: {query}")
    result = await Runner.run(triage_agent, query)
    print(f"Agent: {result.final_output}")
    print(f"  (handled by: {result.last_agent.name})")
    print("-" * 60)

Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_shopping_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Weather Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_weather_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for f

User: What is 17 * 23 + 45?


Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_shopping_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Weather Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_weather_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for f

Agent: To calculate \( 17 \times 23 + 45 \), we can break it down into two parts:

1. First, we calculate \( 17 \times 23 \):
   \[
   17 \times 23 = 391
   \]

2. Next, we add 45 to that result:
   \[
   391 + 45 = 436
   \]

So, the final result is \( 436 \).
  (handled by: Math Expert)
------------------------------------------------------------
User: I need a good keyboard under $100


Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_shopping_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Weather Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_weather_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for f

Agent: I found a mechanical keyboard priced at $89.99. Would you like to know more details about it or explore other options?
  (handled by: Shopping Assistant)
------------------------------------------------------------
User: Should I bring an umbrella to London today?
Agent: Today in London, the temperature is around 14°C, and the weather is foggy. While it might not be raining, the fog can make conditions quite damp. 

**Recommendation:** It's a good idea to bring an umbrella just in case, especially if you're planning to be out for a while. A light jacket or sweater would also be suitable due to the cooler temperature.
  (handled by: Weather Expert)
------------------------------------------------------------


Notice how each query was routed to the appropriate specialist agent, which then used its own tools to answer.

---
## 5. Guardrails

Guardrails let you validate inputs and outputs before/after the agent processes them. If a guardrail **trips**, the agent run is halted and an exception is raised.

- **Input guardrails** — check the user's message before the agent runs
- **Output guardrails** — check the agent's response before returning it

### 5.1 Input Guardrails

In [18]:
from agents import InputGuardrail, GuardrailFunctionOutput, RunContextWrapper


class TopicCheck(BaseModel):
    is_on_topic: bool
    reason: str


topic_checker = Agent(
    name="Topic Checker",
    instructions=(
        "Determine if the user's message is about math, shopping, or weather. "
        "These are the only allowed topics. "
        "Respond with is_on_topic=True if it matches, False otherwise."
    ),
    output_type=TopicCheck,
    model=MODEL,
)


async def check_topic(
    ctx: RunContextWrapper[None], agent: Agent, input: str | list
) -> GuardrailFunctionOutput:
    result = await Runner.run(topic_checker, input, context=ctx.context)
    check = result.final_output_as(TopicCheck)
    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=not check.is_on_topic,
    )


guarded_triage = Agent(
    name="Guarded Triage",
    instructions=triage_agent.instructions,
    handoffs=[math_agent, shopping_agent, weather_agent],
    input_guardrails=[
        InputGuardrail(guardrail_function=check_topic),
    ],
    model=MODEL,
)

print("✓ Guarded triage agent created")

✓ Guarded triage agent created


In [19]:
# On-topic query — should work fine
result = await Runner.run(guarded_triage, "What's 42 * 58?")
print(f"On-topic result: {result.final_output}")

Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_shopping_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Weather Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_weather_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for f

On-topic result: To calculate \( 42 \times 58 \):

1. Multiply 42 by 58.
2. The product is \( 2436 \).

So, \( 42 \times 58 = 2436 \).


In [20]:
# Off-topic query — guardrail should trip
from agents import InputGuardrailTripwireTriggered

try:
    result = await Runner.run(guarded_triage, "Write me a poem about cats")
    print(f"Result: {result.final_output}")
except InputGuardrailTripwireTriggered as e:
    print(f"Guardrail tripped! The query was blocked.")
    print(f"Reason: {e.guardrail_result.output.output_info}")

Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_shopping_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Weather Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_weather_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for f

Guardrail tripped! The query was blocked.
Reason: is_on_topic=False reason='The message is about writing a poem, which is not related to math, shopping, or weather.'


### 5.2 Output Guardrails

Output guardrails validate the agent's response **after** it's generated but **before** it's returned to the user. Useful for content filtering, PII detection, or format validation.

In [21]:
from agents import OutputGuardrail


class OutputCheck(BaseModel):
    contains_disclaimer: bool
    is_appropriate: bool


output_checker = Agent(
    name="Output Checker",
    instructions=(
        "Check if the assistant's response is appropriate and professional. "
        "Set is_appropriate=True if it is suitable for a general audience. "
        "Set contains_disclaimer=True if it includes uncertainty caveats when giving advice."
    ),
    output_type=OutputCheck,
    model=MODEL,
)


async def check_output(
    ctx: RunContextWrapper[None], agent: Agent, output: str
) -> GuardrailFunctionOutput:
    result = await Runner.run(output_checker, output, context=ctx.context)
    check = result.final_output_as(OutputCheck)
    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=not check.is_appropriate,
    )


safe_weather_agent = Agent(
    name="Safe Weather Agent",
    instructions=(
        "You are a weather expert. Provide weather information and advice. "
        "Always include a disclaimer that forecasts may vary."
    ),
    tools=[get_weather_tool],
    output_guardrails=[
        OutputGuardrail(guardrail_function=check_output),
    ],
    model=MODEL,
)

result = await Runner.run(safe_weather_agent, "Will it rain in London?")
print(f"Safe output: {result.final_output}")

Safe output: Currently, it is 14°C and foggy in London. There's no indication of rain at the moment, but weather conditions can change quickly. 

**Disclaimer:** Weather forecasts may vary, so it's always a good idea to check for updates throughout the day.


---
## 6. Agent Context & Shared State

When tools need access to shared data (user info, session state, database connections), use **`RunContextWrapper`** to pass a context object through the entire agent run.

In [22]:
from dataclasses import dataclass, field


@dataclass
class UserSession:
    user_name: str
    user_id: str
    cart: list = field(default_factory=list)
    total: float = 0.0


@function_tool
async def add_to_cart(ctx: RunContextWrapper[UserSession], product_name: str, price: float) -> str:
    """Add a product to the user's shopping cart."""
    ctx.context.cart.append({"name": product_name, "price": price})
    ctx.context.total += price
    return f"Added {product_name} (${price:.2f}) to cart. Cart total: ${ctx.context.total:.2f}"


@function_tool
async def view_cart(ctx: RunContextWrapper[UserSession]) -> str:
    """View the current shopping cart contents."""
    if not ctx.context.cart:
        return f"Cart is empty for {ctx.context.user_name}."
    items = "\n".join(f"  - {item['name']}: ${item['price']:.2f}" for item in ctx.context.cart)
    return f"Cart for {ctx.context.user_name}:\n{items}\nTotal: ${ctx.context.total:.2f}"


@function_tool
async def get_user_info(ctx: RunContextWrapper[UserSession]) -> str:
    """Get information about the current user."""
    return f"User: {ctx.context.user_name} (ID: {ctx.context.user_id})"


shopping_with_state = Agent(
    name="Stateful Shopping Agent",
    instructions=(
        "You are a shopping assistant. Greet the user by name (use get_user_info). "
        "Help them find and add products to their cart. "
        "Use search_products to find items, add_to_cart to add them, "
        "and view_cart to show the current cart."
    ),
    tools=[search_products_tool, add_to_cart, view_cart, get_user_info],
    model=MODEL,
)

print("✓ Stateful shopping agent created")

✓ Stateful shopping agent created


In [23]:
session = UserSession(user_name="Alice", user_id="USR-42")

result = await Runner.run(
    shopping_with_state,
    "Hi! Can you find me some electronics and add the cheapest one to my cart?",
    context=session,
)

print(f"Agent: {result.final_output}")
print(f"\n--- Session State ---")
print(f"Cart: {session.cart}")
print(f"Total: ${session.total:.2f}")

Agent: Hi Alice! I found some electronics for you. The cheapest option, a **Wireless Mouse**, has been added to your cart for **$29.99**. 

If you need anything else, just let me know!

--- Session State ---
Cart: [{'name': 'Wireless Mouse', 'price': 29.99}]
Total: $29.99


---
## 7. Tracing & Observability

The Agents SDK includes built-in **tracing** that records every step of an agent run: LLM calls, tool executions, handoffs, and guardrail checks.

Traces are sent to the [OpenAI dashboard](https://platform.openai.com/traces) by default, where you can inspect them visually.

In [24]:
from agents import trace

with trace("Multi-Agent Demo"):
    result = await Runner.run(
        triage_agent,
        "What's the weather in Paris and how much is 15% tip on $67.50?"
    )
    print(f"Result: {result.final_output}")

print("\n✓ Trace sent to OpenAI dashboard — check https://platform.openai.com/traces")

Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_shopping_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Weather Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_weather_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Math Expert' contains invalid characters for function calling and has been transformed to 'transfer_to_math_expert'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Shopping Assistant' contains invalid characters for f

Result: ### Weather in Paris
- **Temperature:** 18°C
- **Condition:** Partly cloudy

### Tip Calculation
A 15% tip on $67.50 is calculated as follows:
- \( 67.50 \times 0.15 = 10.13 \)

So, the tip amount is **$10.13**. 

If you need clothing recommendations for the weather in Paris, let me know!

✓ Trace sent to OpenAI dashboard — check https://platform.openai.com/traces


You can also disable tracing or use custom trace processors:

```python
from agents import trace, set_tracing_disabled

# Disable tracing entirely
set_tracing_disabled(True)

# Or use a custom trace name for grouping
with trace("Production Pipeline v2"):
    result = await Runner.run(agent, query)
```

---
## 8. Putting It All Together

Let's build a **customer support system** that combines everything: multi-agent handoffs, guardrails, shared context, and structured outputs.

In [25]:
@dataclass
class CustomerContext:
    customer_name: str
    customer_id: str
    membership: str  # "basic", "premium", "enterprise"
    interaction_log: list = field(default_factory=list)


class SupportTicket(BaseModel):
    ticket_id: str
    category: str
    summary: str
    resolution: str
    escalated: bool


@function_tool
async def log_interaction(ctx: RunContextWrapper[CustomerContext], message: str) -> str:
    """Log an interaction note to the customer's record."""
    ctx.context.interaction_log.append(message)
    return f"Logged: {message}"


@function_tool
async def check_membership(ctx: RunContextWrapper[CustomerContext]) -> str:
    """Check the customer's membership tier and benefits."""
    benefits = {
        "basic": "Email support, 5GB storage",
        "premium": "Priority support, 50GB storage, API access",
        "enterprise": "Dedicated support, unlimited storage, SLA guarantee",
    }
    tier = ctx.context.membership
    return f"{ctx.context.customer_name} has {tier} membership: {benefits.get(tier, 'Unknown')}"


# Specialist agents
billing_agent = Agent(
    name="Billing Specialist",
    instructions=(
        "You handle billing and payment questions. "
        "Check membership tier to provide relevant information. "
        "Log all interactions."
    ),
    tools=[check_membership, log_interaction, calculate_tool],
    model=MODEL,
)

technical_agent = Agent(
    name="Technical Support",
    instructions=(
        "You handle technical issues and troubleshooting. "
        "Provide step-by-step solutions. Log all interactions."
    ),
    tools=[log_interaction],
    model=MODEL,
)

# Triage with guardrail
support_triage = Agent(
    name="Support Triage",
    instructions=(
        "You are the front-line support agent. Greet the customer and route to:\n"
        "- Billing Specialist for payment, subscription, pricing questions\n"
        "- Technical Support for bugs, errors, technical issues\n"
        "If unclear, ask a clarifying question before handing off."
    ),
    handoffs=[billing_agent, technical_agent],
    model=MODEL,
)

print("✓ Customer support system ready")

✓ Customer support system ready


In [ ]:
customer = CustomerContext(
    customer_name="Bob",
    customer_id="CUST-789",
    membership="premium",
)

with trace("Customer Support Session"):
    result = await Runner.run(
        support_triage,
        "Hi, I'd like to know what my membership includes and how much an upgrade would cost.",
        context=customer,
    )

print(f"Agent: {result.final_output}")
print(f"Handled by: {result.last_agent.name}")
print(f"\nInteraction log: {customer.interaction_log}")

---
## Summary

| Concept | What It Does | When to Use |
|---|---|---|
| **Function Calling** | Model invokes your functions via structured requests | When the model needs external data or actions |
| **Parallel Tool Calls** | Multiple tools called in a single turn | Independent data lookups |
| **Agentic Loop** | Iterative tool use until task is complete | Complex, multi-step reasoning |
| **Structured Outputs** | Typed Pydantic responses | Data pipelines, APIs, reliable formatting |
| **Agents SDK** | High-level agent abstraction | Any agentic application |
| **Multi-Agent Handoffs** | Specialized agents pass control | Complex domains with distinct expertise |
| **Guardrails** | Input/output validation | Safety, compliance, topic filtering |
| **Agent Context** | Shared state across tools | Stateful conversations, user sessions |
| **Tracing** | Observability for agent runs | Debugging, monitoring, production |

### Further Reading
- [OpenAI Function Calling Guide](https://platform.openai.com/docs/guides/function-calling)
- [OpenAI Agents SDK](https://github.com/openai/openai-agents-python)
- [Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs)